## Setup

In [11]:
%load_ext autoreload
%autoreload 2

from src.dataset import NameDataset, CharCorruptionDataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 01 Data Files

In [3]:
!ls

01_written_questions.ipynb   collect_submission.bat
a4_spr24_student_handout.pdf collect_submission.sh
ass4_debug.ipynb             mingpt-demo
birth_dev.tsv                scripts
birth_places_train.tsv       src
birth_test_inputs.tsv        wiki.txt


In [4]:
!head birth_places_train.tsv

Where was Khatchig Mouradian born?	Lebanon
Where was Jacob Henry Studer born?	Columbus
Where was John Stephen born?	Glasgow
Where was Georgina Willis born?	Australia
Where was Stanley Corrsin born?	Philadelphia
Where was Eduard Ender born?	Rome
Where was John Fisher born?	Glasgow
Where was Iago Dekanozishvili born?	Tbilisi
Where was Lucy Toulmin Smith born?	Boston
Where was George Osborn born?	Rochester


## 02 `NameDataset` 

In [15]:
# Read the data files
with open('wiki.txt', encoding='utf-8') as f:
    wiki_text = f.read()
with open('birth_places_train.tsv', encoding='utf-8') as f:
    birth_places_train = f.read()


# Even if it hasn't been implemented, we use it to define the vocab
corruption_dataset = CharCorruptionDataset(wiki_text, 128)
# Make the name dataset
name_dataset = NameDataset(corruption_dataset, birth_places_train)

block_size: 128
self.data ['Where was Khatchig Mouradian born?\tLebanon', 'Where was Jacob Henry Studer born?\tColumbus', 'Where was John Stephen born?\tGlasgow', 'Where was Georgina Willis born?\tAustralia', 'Where was Stanley Corrsin born?\tPhiladelphia']


In [9]:
name_dataset.stoi[name_dataset.PAD_CHAR]

0

In [16]:
x, y = name_dataset[0]
x, y

x before padding:
Where was Khatchig Mouradian born?⁇Lebanon⁇
length of x before padding: 43
x after padding:
Where was Khatchig Mouradian born?⁇Lebanon⁇□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□
length of x after padding: 128
y after padding:
□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□⁇Lebanon⁇□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□
length of y after padding: 127
x after removing last char:
Where was Khatchig Mouradian born?⁇Lebanon⁇□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□
length of x after removing last char: 127


(tensor([51, 63, 60, 73, 60,  3, 78, 56, 74,  3, 39, 63, 56, 75, 58, 63, 64, 62,
          3, 41, 70, 76, 73, 56, 59, 64, 56, 69,  3, 57, 70, 73, 69, 28,  1, 40,
         60, 57, 56, 69, 70, 69,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0]),
 tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1, 40, 60,
         57, 56, 69, 70, 69,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0, 

## 03 (c) Implement finetuning (without pretraining)

In [10]:
!python src/run.py finetune vanilla wiki.txt \
    --writing_params_path vanilla.model.params \
    --finetune_corpus_path birth_places_train.tsv

Device: mps
Number of parameters: 3.32M
Model on device:  mps:0
Finetuning with writing params path: vanilla.model.params
epoch 1 iter 7: train loss 3.47418. lr 5.999844e-04: 100%|█| 8/8 [00:49<00:00,  
epoch 2 iter 5: train loss 2.97700. lr 5.999496e-04:  75%|▊| 6/8 [01:32<00:42, 2^C


## (e) (10 points) Define a span corruption function for pretraining.

In [16]:
# Read the data files
with open('wiki.txt', encoding='utf-8') as f:
    wiki_text = f.read()

# Even if it hasn't been implemented, we use it to define the vocab
corruption_dataset = CharCorruptionDataset(data=wiki_text)

Unique characters in data: 254
data has 418,351 characters, 256 unique.
Number of lines in data: 2,937
First few lines of data: ['Khatchig Mouradian. Khatchig Mouradian is a journalist, writer and translator born in Lebanon .', 'Jacob Henry Studer. Jacob Henry Studer (26 February 1840 Columbus, Ohio - 2 August 1904 New York City) was a printer, lithographer, painter, and popular ornithologist active in Columbus, Ohio from the 1860s to the 1880s .', "John Stephen. Born in Glasgow, Stephen became a welder's apprentice on leaving school .", 'Georgina Willis. Georgina Willis is an award winning film director who was born in Australia and now lives in London .', 'Stanley Corrsin. Corrsin was born on 3 April 1920 in Philadelphia, Pennsylvania .']
Last line: Gerald Palmer. Although he was born in England, Gerald Palmer grew up in Southern Rhodesia, now Zimbabwe, where his father was chief engineer to the state run railways .


In [25]:
document = corruption_dataset[0]
print(f'Document: {document}')

Document at index 0: Khatchig Mouradian. Khatchig Mouradian is a journalist, writer and translator born in Lebanon .
Truncated document: Khatchig Mouradian. Khatchig Mouradian is a journalist, writer and t
[Khatchig Mouradian. Khatchig M] [ouradian is a jou] [rnalist, writer and t]
Rearranged document: Khatchig Mouradian. Khatchig M⁇rnalist, writer and t⁇ouradian is a jou□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□
Document: (0, 0)
